# III-08. 색등급도 그리기

「지구과학 데이터 사이언스」 책의 코드입니다. 설명은 책을 참고하세요.

**준비** (책에는 싣지 않은 공통 셀)

In [ ]:
# 필요한 모듈 정의 (버전 명시 가능)
packages_str = "numpy, pandas, matplotlib, requests, version_information"

In [ ]:
# 공통 함수(dses_utils) 불러오기 — 코랩/로컬 모두 동작(별도 설치 불필요)
import os, urllib.request
try:
    urllib.request.urlretrieve("https://raw.githubusercontent.com/Kiehyun/dses_utils/main/dses_utils.py", "dses_utils.py")
except Exception:
    pass  # 인터넷이 안 되면 기존에 받아둔 dses_utils.py 사용
import dses_utils as du
du.check_and_install(packages_str)
# 한글 폰트 설정 — 공통 함수(dses_utils.set_korean_font)로 통일
# (반복되던 OS별 rcParams 설정 코드는 dses_utils.py 로 옮겼습니다)
import dses_utils as du
du.set_korean_font()
# 프로젝트(자료 저장) 폴더 생성 — 공통 함수 사용
proj_dir = du.make_project_dir(du.current_notebook_name())

**code 105.**

In [ ]:
from pathlib import Path

# (HLA 데이터셋 이름, 파일 이름, 구글 드라이브 사본 ID)
files = [
('hst_11233_06_wfpc2_f814w_f450w_wf', 'hst_11233_06_wfpc2_multiwave_wf_daophot_trm.cat', '1HX3_8X4vygQG_bj9107m9DNhIBl8BMc3'),
('hst_11233_06_wfpc2_f450w_wf', 'hst_11233_06_wfpc2_f450w_wf_daophot_trm.cat', '1plnoVATBy9lXG5Yac69ofDl4kiLTCKuv'),
('hst_11233_06_wfpc2_f814w_wf', 'hst_11233_06_wfpc2_f814w_wf_daophot_trm.cat', '1Yhe-LMiYMJpuGnAv7G43jevDZCoeml7P')
]

for dataset, fname, fid in files:
    url = f"https://hla.stsci.edu/cgi-bin/getdata.cgi?dataset={dataset}&filename={fname}"
    try:  # 허블 레거시 아카이브에서 직접 받기
        du.download(url, Path(proj_dir) / fname)
    except Exception:  # 실패하면 구글 드라이브 사본
        du.download(f"https://docs.google.com/uc?export=download&id={fid}", Path(proj_dir) / fname)

# 이름순: [0] f450w, [1] f814w, [2] multiwave
fpaths = sorted(Path(proj_dir).glob('*.cat'))

**code 106.**

In [ ]:
# 파일 앞 40줄 출력(공통 함수, 인코딩 자동 판별)
head = du.preview_text(fpaths[2], n=40)  # 출력한 줄 목록은 head에 담아 둠

**code 107.**

In [ ]:
import numpy as np

# 데이터가 시작되는 줄 수(파일을 열어 확인)
skiprows = 35

DAO_array = np.loadtxt(
    f'{fpaths[2]}',
    comments='#',
    dtype=float,
    delimiter=None,  # 공백 여러 칸도 구분자
    skiprows=skiprows
)

print("type(DAO_array):", type(DAO_array))
print("DAO_array.shape:", DAO_array.shape)

**code 108.**

In [ ]:
# 행 방향 슬라이싱
print("DAO_array[1, :]: {}".format(DAO_array[1, :]))

# 열 방향 슬라이싱
print("DAO_array[:, 1]: {}".format(DAO_array[:, 1]))

print("CI:\n {}".format(DAO_array[:,5]-DAO_array[:,6]))

**code 109.**

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# 색등급도(CMD)
plt.plot(DAO_array[:,5]-DAO_array[:,6], DAO_array[:,5], 'o', ms=2, alpha=0.2)

# 등급 축 뒤집기(밝을수록 위)
plt.gca().invert_yaxis()

plt.show()

**code 110.**

In [ ]:
print("DAO_array[:,5].min(): {}".format(DAO_array[:,5].min()))
print("DAO_array[:,5].max(): {}".format(DAO_array[:,5].max()))
print("DAO_array[:,6].min(): {}".format(DAO_array[:,6].min()))
print("DAO_array[:,6].max(): {}".format(DAO_array[:,6].max()))
print("DAO_array[:,7].min(): {}".format(DAO_array[:,7].min()))
print("DAO_array[:,7].max(): {}".format(DAO_array[:,7].max()))
print("DAO_array[:,8].min(): {}".format(DAO_array[:,8].min()))
print("DAO_array[:,8].max(): {}".format(DAO_array[:,8].max()))

import numpy as np

num1 = np.arange(0, 11, 2)                   # [0, 2, 4, 6, 8, 10]
print("num1's elements:\n {}".format(num1))

# 마스킹 예
tf   = np.array([False, True, False, True, True, False])

print("tf:\n {}".format(tf))

print("num1's selected elements:\n {}".format(num1[tf]))  # tf가 True인 자리만

# 실제 자료에 마스킹
mask  = np.zeros(len(DAO_array)).astype(bool)

# -999는 측정 실패 표시
for i in range(0, len(DAO_array)):
    # 5~10번 열 중 하나라도 -999이면 제외
    if DAO_array[i, 5] == -999 or DAO_array[i, 6] == -999 \
        or DAO_array[i, 7] == -999 or DAO_array[i, 8] == -999 \
        or DAO_array[i, 9] == -999 or DAO_array[i, 10] == -999:
        mask[i]  = False
    else :
        mask[i]  = True
print("mask:\n {}".format(mask))
# 유효한 행만
DAO_array_masked = DAO_array[mask]      
print("DAO_array_masked.shape: {}".format(DAO_array_masked.shape))
print("DAO_array_masked[:5]:\n {}".format(DAO_array_masked[:5]))

**code 111.**

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# 색등급도(CMD)
plt.plot(DAO_array[mask,5]-DAO_array[mask,6], DAO_array[mask,5], 'o', ms=2, alpha=0.2)

# 등급 축 뒤집기(밝을수록 위)
plt.gca().invert_yaxis()

plt.show()

**code 112.**

In [ ]:
import numpy as np

ch1, ch2 = "F450", "F814"
data_type  = "WFPC2_WFC"
object_name = 'M80'

import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(10.5,8))

ax.plot(DAO_array[mask,5]-DAO_array[mask,6], DAO_array[mask,5], 'o', ms=2, alpha=0.2)

ax.set_xlabel('색지수 ({} − {})'.format(ch1, ch2), fontdict={'fontsize':14})
ax.set_ylabel('등급 ({})'.format(ch1), fontdict={'fontsize':14})

ax.set_title('{}의 색등급도 ({}, MagAP2)'.format(object_name, data_type),
             fontdict={'fontsize':16})

ax.grid()
plt.gca().invert_yaxis()  # 등급 축 뒤집기

plt.annotate('허블 우주 망원경 자료(HLA)',
        xy=(1, 0), xytext=(0, -50), va='top', ha='right',
        xycoords='axes fraction', textcoords='offset points')

plt.savefig(f'{proj_dir}/{object_name}_{data_type}_CMD.png',
            dpi=300, bbox_inches='tight')
print(f'{proj_dir}/{object_name}_{data_type}_CMD.png 파일을 만들었습니다.')

plt.show()

**code 113.**

In [ ]:
ch1, ch2 = "F450", "F814"
data_type = "WFPC2_WFC"
object_name = 'M80'

# -999(결측) 행 제외
for i in range(0, len(DAO_array)):
    if DAO_array[i, 5] == -999 or DAO_array[i, 6] == -999\
        or DAO_array[i, 7] == -999 or DAO_array[i, 8] == -999:
        mask[i] = False
    else:
        mask[i] = True

data_type1 = "WFPC2_MagAP2"
data_type2 = "WFPC2_TotMag"

fig = plt.figure(figsize=(12.0, 6.0))
ax1 = fig.add_subplot(121)
ax2 = fig.add_subplot(122)

# 왼쪽: MagAP2 등급(5 · 6번 열), 오른쪽: TotMag 등급(7 · 8번 열)
ax1.plot(DAO_array[mask,5] - DAO_array[mask,6], DAO_array[mask,5], 'o', ms=2, alpha=0.2)
ax2.plot(DAO_array[mask,7] - DAO_array[mask,8], DAO_array[mask,7], 'o', ms=2, alpha=0.2)

ax1.set_xlabel(f'색지수 ({ch1} − {ch2})', fontdict={'fontsize':14})
ax1.set_ylabel(f'등급 ({ch1})', fontdict={'fontsize':14})
ax2.set_xlabel(f'색지수 ({ch1} − {ch2})', fontdict={'fontsize':14})
ax2.set_ylabel(f'등급 ({ch1})', fontdict={'fontsize':14})

ax1.invert_yaxis()
ax2.invert_yaxis()
ax1.grid()
ax2.grid()

ax1.set_title(f'{object_name}의 색등급도 ({data_type1})',
             fontdict={'fontsize':16})
ax2.set_title(f'{object_name}의 색등급도 ({data_type2})',
             fontdict={'fontsize':16})

plt.annotate('허블 우주 망원경 자료(HLA)',
        xy=(1, 0), xytext=(0, -50), va='top', ha='right',
        xycoords='axes fraction', textcoords='offset points')

plt.savefig(f'{proj_dir}/{object_name}_{data_type1}_{data_type2}_CMD.png',
            dpi=300, bbox_inches='tight')
print(f'{proj_dir}/{object_name}_{data_type1}_{data_type2}_CMD.png 파일을 만들었습니다.')

plt.show()

**code 114.**

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

file_path = f'{fpaths[2]}'  # multiwave 파일
skiprows = 35               # 앞 설명 줄 수

df = pd.read_csv(
    file_path,
    sep=r'\s+',   # 공백 여러 칸이 구분자
    comment='#',  # '#' 줄은 건너뛰기
    header=None,
    skiprows=skiprows
)

# 5~8번 열에 -999가 있는 행 제외
mask = ~(
    (df[5] == -999) | (df[6] == -999) |
    (df[7] == -999) | (df[8] == -999)
)
df_masked = df[mask]

ch1, ch2 = "F450", "F814"     # 필터 이름
data_type1  = "WFPC2 MagAP2"
data_type2  = "WFPC2 TotMag"
object_name = 'M80'

fig = plt.figure(figsize=(12.0, 6.0))
ax1 = fig.add_subplot(121)
ax2 = fig.add_subplot(122)

# 왼쪽: MagAP2 등급
ax1.plot(df_masked[5] - df_masked[6], df_masked[5], 'o', ms=2, alpha=0.2)
ax1.set_xlabel(f'색지수 ({ch1} − {ch2})', fontsize=14)
ax1.set_ylabel(f'등급 ({ch1})', fontsize=14)
ax1.set_title(f'{object_name}의 색등급도 ({data_type1})', fontsize=16)
ax1.invert_yaxis()                                 # 등급이 작을수록 밝으므로 y축 반전
ax1.grid()

# 오른쪽: TotMag 등급
ax2.plot(df_masked[7] - df_masked[8], df_masked[7], 'o', ms=2, alpha=0.2)
ax2.set_xlabel(f'색지수 ({ch1} − {ch2})', fontsize=14)
ax2.set_ylabel(f'등급 ({ch1})', fontsize=14)
ax2.set_title(f'{object_name}의 색등급도 ({data_type2})', fontsize=16)
ax2.invert_yaxis()
ax2.grid()

# 아래쪽에 출처 표기
plt.annotate('허블 우주 망원경 자료(HLA)',
        xy=(1, 0), xytext=(0, -50), va='top', ha='right',
        xycoords='axes fraction', textcoords='offset points')

plt.show()

**code 115.**

In [ ]:
# F450W 파일 앞 65줄 출력(F814W 파일도 구조가 같음)
head = du.preview_text(fpaths[0], n=65)

**code 116.**

In [ ]:
import numpy as np

# 데이터가 시작되는 줄 수(파일을 열어 확인)
skiprows = 60

F450_array = np.loadtxt(
    f'{fpaths[0]}',
    comments='#',
    dtype=float,
    delimiter=None,  # 공백 여러 칸도 구분자
    skiprows=skiprows
)

F814_array = np.loadtxt(
    f'{fpaths[1]}', comments='#', dtype=float, delimiter=None,
    skiprows=skiprows
)
print("type(F450_array):", type(F450_array))
print("type(F814_array):", type(F814_array))
print("F450_array[1, :]: {}".format(F450_array[1, :]))
print("F814_array[1, :]: {}".format(F814_array[1, :]))  

**code 117.**

In [ ]:
print("F450.shape: {}".format(F450_array.shape))
print("F814.shape: {}".format(F814_array.shape))
print("F450[:,4:6]: {}".format(F450_array[:,4:6]))
print("F814[:,4:6]: {}".format(F814_array[:,4:6]))

**code 118.**

In [ ]:
import numpy as np

mag_col = 7  # MagAP2(0.30초각 조리개) 등급 열 — multiwave 파일의 그림과 같은 조리개
mask = np.zeros(len(F450_array)).astype(bool)  # True: 쓸 별
color = np.zeros(len(F450_array))              # 색지수

# 두 필터 목록에서 ID가 같은 별 짝짓기
for i in range(0, len(F450_array)):
    ID_ch1 = F450_array[i, 4]  # 4번 열: 별 ID
    for j in range(0, len(F814_array)):
        ID_ch2 = F814_array[j, 4]
        if ID_ch1 == ID_ch2:
            # -999(측정 실패)이면 쓰지 않음
            if F450_array[i, mag_col] != -999 and F814_array[j, mag_col] != -999:
                mask[i] = True
                # 색지수 = F450 등급 - F814 등급
                color[i] = F450_array[i, mag_col] - F814_array[j, mag_col]
            break  # 찾으면 다음 별로

print("len(color):{}".format(len(color)))
print("len(F450[mask]):{}".format(len(F450_array[mask])))
print("len(color[mask]):{}".format(len(color[mask])))

**code 119.**

In [ ]:
color[mask]

**code 120.**

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# 색등급도(CMD)
plt.plot(color[mask], F450_array[mask, mag_col], 'o', ms=2, alpha=0.2)

# 등급 축 뒤집기(밝을수록 위)
plt.gca().invert_yaxis()

plt.show()